# 46 · k-means com k = 4 nas três versões de variáveis

Mesma população (13.999 assinantes ativos que treinaram nos 90 dias até 30/09/2026), mesma semente e
mesmo preparo das variáveis dos notebooks 44 e 45. Muda só o k, de 3 para 4, rodado nas três versões:

| Versão | Variáveis |
|---|---|
| **v1** | as oito: treinos por semana, dias sem treino, tempo de casa, % ao ar livre, km por treino, engajamento (z) + plano e 6 canais (0/1) |
| **v1 sem canal** | a v1 sem os 6 indicadores de canal |
| **v2** | a v1 sem canal e sem engajamento |

Para cada versão: inércia, silhueta, estabilidade (5 sementes e 5 reamostragens), quanto cada variável
separa, perfil dos grupos e a comparação com o k = 3 da mesma versão. Ordem de execução: 39 → 41 → 42 → 43 → 44 → 45 → 46.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
K = 4; SEED = 0
AR_LIVRE = ["corrida", "caminhada", "pedal"]

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master:", m.shape)
print(m.segmento_regra.value_counts().to_string())
e = m[m.segmento_regra == "elegível ao k-means"].copy().reset_index(drop=True)
assert (e.data_referencia == CORTE).all() and e.ativa_no_corte.all()
print("\nelegíveis ao k-means:", len(e), "| contas distintas:", e.user_id.nunique())

tabela master: (63929, 119)
segmento_regra
fora do escopo (não é assinante ativo)    49176
elegível ao k-means                       13999
inativo em 90 dias                          754

elegíveis ao k-means: 13999 | contas distintas: 13999


## 1. % dos treinos ao ar livre

In [2]:
e["pct_ar_livre"] = (e[["treinos_" + t for t in AR_LIVRE]].sum(axis=1) / e.treinos_90d * 100).astype(float)
print("treinos_90d > 0 em todos os elegíveis:", bool((e.treinos_90d > 0).all()))
print(e.pct_ar_livre.describe().round(1).to_dict())
print("0% ao ar livre:", int((e.pct_ar_livre == 0).sum()), "| 100% ao ar livre:", int((e.pct_ar_livre == 100).sum()))

treinos_90d > 0 em todos os elegíveis: True
{'count': 13999.0, 'mean': 54.4, 'std': 34.6, 'min': 0.0, '25%': 16.7, '50%': 71.4, '75%': 83.3, 'max': 100.0}
0% ao ar livre: 691 | 100% ao ar livre: 1087


## 2. Km por treino

Média de `distancia_km` dos treinos ao ar livre (corrida, caminhada, pedal) na janela de 90 dias até o
corte (mesma janela da tabela master: 0 a 89 dias antes de 30/09/2026), só treinos com GPS válido e
distância maior que zero. Quem não tem nenhum treino com distância na janela fica com **0 km**.

In [3]:
a = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "distancia_km", "gps_ok"])
print("treinos no arquivo:", len(a))
a = a[a.user_id.isin(e.user_id)]
print("-> dos elegíveis:", len(a))
a = a[(CORTE - a.data).dt.days.between(0, 89)]
print("-> na janela de 90 dias:", len(a))
al = a[a.tipo.isin(AR_LIVRE)]
print("-> ao ar livre:", len(al), "| GPS falhou:", int((~al.gps_ok).sum()), "| GPS ok sem distância ou zero:", int((al.gps_ok & ~(al.distancia_km > 0)).sum()))
d = al[al.gps_ok & (al.distancia_km > 0)]
print("-> com distância válida:", len(d), "| contas:", d.user_id.nunique())
km = d.groupby("user_id").distancia_km.mean().rename("km_por_treino")
e = e.merge(km, on="user_id", how="left")
fal = set(al.user_id) - set(d.user_id)
print("\ncontas sem treino com distância (ficam com 0 km):", int(e.km_por_treino.isna().sum()))
print("  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou):", len(fal))
e["km_por_treino"] = e.km_por_treino.fillna(0)
e["km_por_treino_log"] = np.log1p(e.km_por_treino)
print("\nkm por treino, quem tem distância:", e.km_por_treino[e.km_por_treino > 0].describe().round(1).to_dict())
print("km por treino por tipo principal (mediana, quem tem distância):")
print(e[e.km_por_treino > 0].groupby("tipo_principal").km_por_treino.median().round(1).to_string())
print("assimetria antes e depois do log:", round(e.km_por_treino.skew(), 2), "->", round(e.km_por_treino_log.skew(), 2))

treinos no arquivo: 1252488


-> dos elegíveis:

 941195
-> na janela de 90 dias: 372012
-> ao ar livre: 199229 | GPS falhou: 0 | GPS ok sem distância ou zero: 0
-> com distância válida: 199229 | contas: 13308

contas sem treino com distância (ficam com 0 km): 691
  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou): 0

km por treino, quem tem distância: {'count': 13308.0, 'mean': 8.7, 'std': 9.5, 'min': 1.9, '25%': 5.0, '50%': 5.8, '75%': 7.6, 'max': 162.8}
km por treino por tipo principal (mediana, quem tem distância):
tipo_principal
caminhada    5.40
corrida      7.00
forca        5.20
funcional    5.20
pedal       27.30
yoga         5.10
assimetria antes e depois do log: 6.28 -> -0.17


## 3. Engajamento, plano e canal

In [4]:
ORD = {"possível churn": 0, "baixo": 1, "médio": 2, "alto": 3}
e["engajamento_ord"] = e.nivel_engajamento.map(ORD)
assert e.engajamento_ord.notna().all()
print(e.nivel_engajamento.value_counts().to_string())
e["plano_anual"] = (e.plano == "anual").astype(int)
print("\nplano anual:", int(e.plano_anual.sum()), f"({e.plano_anual.mean():.1%})")
canal = pd.get_dummies(e.canal_aquisicao, prefix="canal").astype(int)
e = pd.concat([e, canal], axis=1)
CANAIS = list(canal.columns)
print("\ncanal:"); print(e.canal_aquisicao.value_counts().to_string())

nivel_engajamento
alto              10689
médio              2366
possível churn      579
baixo               365

plano anual: 7727 (55.2%)



canal:
canal_aquisicao
organico                4410
influenciador           2379
meta_ads                2294
indicacao               2113
google_ads              1435
parceria_corporativa    1368


## 4. As três matrizes

In [5]:
NUM_V1 = ["treinos_por_semana", "dias_sem_treinar_log", "duracao_dias_log", "pct_ar_livre", "km_por_treino_log", "engajamento_ord"]
NUM_V2 = NUM_V1[:-1]
def matriz(num, ind):
    sc = StandardScaler().fit(e[num])
    return pd.concat([pd.DataFrame(sc.transform(e[num]), columns=[c + "_z" for c in num]), e[ind].reset_index(drop=True)], axis=1)
VERSOES = {
    "v1": matriz(NUM_V1, ["plano_anual"] + CANAIS),
    "v1_sem_canal": matriz(NUM_V1, ["plano_anual"]),
    "v2": matriz(NUM_V2, ["plano_anual"]),
}
for n, X in VERSOES.items(): print(n, X.shape, "| vazios:", int(X.isna().sum().sum()))

v1 (13999, 13) | vazios: 0
v1_sem_canal (13999, 7) | vazios: 0
v2 (13999, 6) | vazios: 0


## 5. k-means com k = 4 e as lentes de avaliação

Grupos numerados por tamanho (1 = maior). Silhueta com todos os pontos.

In [6]:
def r2(col, lab):
    tot = ((col - col.mean()) ** 2).sum()
    return sum(len(s) * (s.mean() - col.mean()) ** 2 for _, s in col.groupby(lab)) / tot if tot else np.nan

res, rot, sep = [], {}, {}
for n, X in VERSOES.items():
    km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
    lab = km.labels_
    mapa = {g: i + 1 for i, g in enumerate(pd.Series(lab).value_counts().index)}
    rot[n] = pd.Series(lab).map(mapa).values
    seeds = [adjusted_rand_score(lab, KMeans(n_clusters=K, n_init=10, random_state=s).fit_predict(X)) for s in [1, 2, 3, 4, 5]]
    rng = np.random.default_rng(SEED); boot = []
    for i in range(5):
        idx = rng.integers(0, len(X), len(X))
        boot.append(adjusted_rand_score(lab, KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X.iloc[idx]).predict(X)))
    sep[n] = pd.Series({c: r2(X[c], lab) for c in X.columns})
    res.append({"versao": n, "inercia": km.inertia_, "voltas": km.n_iter_, "silhueta": silhouette_score(X, lab),
                "rand_sementes_min": min(seeds), "rand_reamostra_media": np.mean(boot), "rand_reamostra_min": min(boot),
                "tamanhos": sorted(np.bincount(lab).tolist(), reverse=True)})
res = pd.DataFrame(res).set_index("versao")
print(res.round(3).to_string())
print("\nquanto cada variável separa os grupos (R² entre grupos):")
print(pd.DataFrame(sep).round(3).sort_values("v1", ascending=False).to_string())

               inercia  voltas  silhueta  rand_sementes_min  rand_reamostra_media  rand_reamostra_min                  tamanhos
versao                                                                                                                         
v1           58,086.25      13      0.19               0.99                  0.98                0.96  [5128, 4169, 2695, 2007]
v1_sem_canal 46,892.14      20      0.24               1.00                  0.98                0.97  [5098, 4190, 2684, 2027]
v2           41,521.03      16      0.24               0.99                  0.92                0.68  [4732, 4521, 2703, 2043]

quanto cada variável separa os grupos (R² entre grupos):
                             v1  v1_sem_canal   v2
engajamento_ord_z          0.68          0.68  NaN
pct_ar_livre_z             0.62          0.63 0.76
dias_sem_treinar_log_z     0.60          0.60 0.43
duracao_dias_log_z         0.53          0.53 0.52
treinos_por_semana_z       0.33          0.33 0

### Comparação com o k = 3 da mesma versão

Linhas: grupo com k = 3; colunas: grupo com k = 4. Mostra qual grupo do k = 3 foi dividido.
A v1 sem canal é comparada com os rótulos da v1 (k = 3), que são praticamente os mesmos (Rand 0,998, notebook 44).

In [7]:
k3_v1 = pd.read_parquet(P / "kmeans_k3_rotulos.parquet", columns=["user_id", "grupo_k3"])
k3_v2 = pd.read_parquet(P / "kmeans_k3_v2_rotulos.parquet", columns=["user_id", "grupo_k3"])
for n, ref in [("v1", k3_v1), ("v1_sem_canal", k3_v1), ("v2", k3_v2)]:
    c = e[["user_id"]].assign(k4=rot[n]).merge(ref, on="user_id", how="left")
    assert c.grupo_k3.notna().all()
    print(f"--- {n}: Rand ajustado k = 3 x k = 4:", round(adjusted_rand_score(c.grupo_k3, c.k4), 3))
    print(pd.crosstab(c.grupo_k3, c.k4, rownames=["k = 3"], colnames=["k = 4"], margins=True)); print()

--- v1: Rand ajustado k = 3 x k = 4: 0.746
k = 4     1     2     3     4    All
k = 3                               
1      5109     3     4  1140   6256
2         0  4155    47   742   4944
3        19    11  2644   125   2799
All    5128  4169  2695  2007  13999

--- v1_sem_canal: Rand ajustado k = 3 x k = 4: 0.743
k = 4     1     2     3     4    All
k = 3                               
1      5079     3     5  1169   6256
2         0  4171    45   728   4944
3        19    16  2634   130   2799
All    5098  4190  2684  2027  13999

--- v2: Rand ajustado k = 3 x k = 4: 0.708
k = 4     1     2     3     4    All
k = 3                               
1      4620     1    57   684   5362
2       107  4285    79   260   4731
3         5   235  2567  1099   3906
All    4732  4521  2703  2043  13999



## 6. Perfil dos grupos em cada versão

Medianas na unidade original e a lente de três fatores: ativos (tamanho), engajamento (níveis) e sinal de
saída (renovação automática desligada no corte; não há saída observada depois de 30/09/2026).

In [8]:
def perfil(gr):
    g = e.assign(grupo=gr).groupby("grupo")
    p = pd.DataFrame({
        "assinantes": g.size(), "pct_base": g.size() / len(e) * 100,
        "treinos_por_semana": g.treinos_por_semana.median(), "dias_sem_treinar": g.dias_sem_treinar.median(),
        "meses_de_casa": g.meses_de_casa.median(), "pct_ar_livre": g.pct_ar_livre.median(),
        "km_por_treino": g.km_por_treino.median(), "pct_anual": g.plano_anual.mean() * 100,
        "pct_alto": g.nivel_engajamento.apply(lambda s: (s == "alto").mean() * 100),
        "pct_medio": g.nivel_engajamento.apply(lambda s: (s == "médio").mean() * 100),
        "pct_baixo": g.nivel_engajamento.apply(lambda s: (s == "baixo").mean() * 100),
        "pct_possivel_churn": g.nivel_engajamento.apply(lambda s: (s == "possível churn").mean() * 100),
        "pct_renovacao_desligada": g.renovacao_desligada.apply(lambda s: s.astype(float).mean() * 100),
        "tipo_principal_mais_comum": g.tipo_principal.agg(lambda s: s.value_counts().index[0]),
    })
    return p
PERF = {}
for n in VERSOES:
    PERF[n] = perfil(rot[n])
    print(f"===== {n}"); print(PERF[n].round(1).T); print()
print("base: renovação desligada", round(e.renovacao_desligada.astype(float).mean() * 100, 1), "% | engajamento alto", round((e.nivel_engajamento == "alto").mean() * 100, 1), "%")

===== v1
grupo                            1          2        3        4
assinantes                    5128       4169     2695     2007
pct_base                     36.60      29.80    19.30    14.30
treinos_por_semana            2.60       2.30     0.50     0.90
dias_sem_treinar              1.00       1.00    20.00     2.00
meses_de_casa                10.60      10.40    10.00     0.90
pct_ar_livre                 80.80      14.30    77.80    68.20
km_por_treino                 7.10       5.20     6.00     5.60
pct_anual                    53.70      50.30    80.50    35.20
pct_alto                     98.00      92.00     0.00    90.90
pct_medio                     2.00       8.00    65.00     9.10
pct_baixo                     0.00       0.00    13.50     0.00
pct_possivel_churn            0.00       0.00    21.50     0.00
pct_renovacao_desligada       5.10       5.10    54.10     2.80
tipo_principal_mais_comum  corrida  funcional  corrida  corrida

===== v1_sem_canal
grupo      

### Canal de aquisição por grupo na v1 (a única versão em que o canal entra)

In [9]:
print((pd.crosstab(rot["v1"], e.canal_aquisicao, normalize="index") * 100).round(1))

canal_aquisicao  google_ads  indicacao  influenciador  meta_ads  organico  parceria_corporativa
row_0                                                                                          
1                     10.70      15.70          15.10     16.60     32.00                  9.90
2                     10.60      15.20          14.80     16.10     32.60                 10.70
3                     10.60      14.60          15.30     18.30     32.00                  9.20
4                      8.00      13.90          28.60     13.80     27.30                  8.40


## 7. Salvar

In [10]:
out = e[["user_id"]].assign(grupo_k4_v1=rot["v1"], grupo_k4_v1_sem_canal=rot["v1_sem_canal"], grupo_k4_v2=rot["v2"])
out.to_parquet(P / "kmeans_k4_rotulos.parquet", index=False)
res.assign(tamanhos=res.tamanhos.astype(str)).round(4).to_csv(S / "kmeans_k4_metricas.csv", encoding="utf-8-sig")
pd.concat(PERF, names=["versao", "grupo"]).round(2).to_csv(S / "kmeans_k4_perfis.csv", encoding="utf-8-sig")
pd.DataFrame(sep).round(4).to_csv(S / "kmeans_k4_separacao_por_variavel.csv", encoding="utf-8-sig", index_label="variavel")
print("rótulos:", out.shape, "-> dados/processados/kmeans_k4_rotulos.parquet")
print("saídas: kmeans_k4_metricas.csv, kmeans_k4_perfis.csv, kmeans_k4_separacao_por_variavel.csv")

rótulos: (13999, 4) -> dados/processados/kmeans_k4_rotulos.parquet
saídas: kmeans_k4_metricas.csv, kmeans_k4_perfis.csv, kmeans_k4_separacao_por_variavel.csv
